
%md

## CELEBAL TECHNOLOGIES

#### Data Engineering Internship (CEI Program 2026) Week 5 Assignment

#### Apache Spark: DataFrames, Transformations & Data Cleaning
####File - Advance Transformation 
---

### Submitted By
**Harshit Sharma**  
*Data Engineering Intern*

---


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### Transformation 1: Load Dataset

Used `spark.read.table()` to load the sales dataset from the Unity Catalog into a Spark DataFrame. This is the initial step that makes the data available for further cleaning, transformation, and analysis.

**Operation Used:** `spark.read.table()`

In [0]:
df = spark.read.table('practice_catalog.new_source.sales')
df.limit(5).display()

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


### Transformation 2: Select Required Columns

Used the `select()` transformation to extract only the relevant columns needed for analysis. This helps reduce unnecessary data processing, improves readability, and focuses the dataset on business-critical attributes.

**Operation Used:** `select()`

In [0]:
df_selected = df.select(
    "Item_Identifier",
    "Item_Weight",
    "Item_Fat_Content",
    "Item_Visibility",
    "Item_Type",
    "Item_MRP",
    "Outlet_Identifier",
    "Outlet_Establishment_Year",
    "Outlet_Size",
    "Outlet_Location_Type",
    "Outlet_Type",
    "Item_Outlet_Sales"
)
df_selected.limit(5).display()

Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


### Transformation 3: Rename Columns

Used `withColumnRenamed()` to rename selected columns with a consistent lowercase naming convention. This improves readability, simplifies query writing, and follows standard data engineering practices.

**Operations Used:** `withColumnRenamed()`

In [0]:
df_renamed = (
    df_selected
    .withColumnRenamed("Item_Weight", "item_weight")
    .withColumnRenamed("Item_Fat_Content", "item_fat_content")
    .withColumnRenamed("Item_Visibility", "item_visibility")
    .withColumnRenamed("Item_Type", "item_type")
    .withColumnRenamed("Item_MRP", "item_mrp")
)

df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: long (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



### Transformation 4: Remove Duplicate Records

Used the `dropDuplicates()` transformation to eliminate duplicate rows from the dataset. This ensures data consistency, prevents duplicate records from affecting analysis, and improves the accuracy of business insights.

**Operation Used:** `dropDuplicates()`

In [0]:
df_dedup = df_renamed.dropDuplicates()
print('Total Removes duplicate records. ',df_dedup.count())

Total Removes duplicate records.  8523


### Transformation 5: Filter Records

Used the `filter()` transformation to retain only records where the item price (`item_mrp`) is greater than 100. This helps focus the analysis on products with higher market value and removes low-priced items from the dataset.

**Operation Used:** `filter()`

In [0]:
df_filtered = df_dedup.filter(df_dedup.item_mrp > 100)
df_filtered.limit(5).display()

Item_Identifier,item_weight,item_fat_content,item_visibility,item_type,item_mrp,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,null,Tier 3,Grocery Store,732.38
FDP10,null,Low Fat,0.127469857,Snack Foods,107.7622,OUT027,1985,Medium,Tier 3,Supermarket Type3,4022.7636
FDU28,19.2,Regular,0.09444959,Frozen Foods,187.8214,OUT017,2007,null,Tier 2,Supermarket Type1,4710.535


%md
### Transformation 6: Analyze Missing Values

Used `isNull()`, `when()`, and `sum()` to identify missing values in each column of the dataset. This step helps understand data quality issues before applying data cleaning techniques.

**Operations Used:** `isNull()`, `when()`, `sum()`

In [0]:
# check which column have null values
df_dedup.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_dedup.columns
]).display()

Item_Identifier,item_weight,item_fat_content,item_visibility,item_type,item_mrp,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,1463,0,0,0,0,0,0,2410,0,0,0


In [0]:
# Calculate average weight
avg_weight = df_dedup.select(avg("Item_Weight")).first()[0]

# Filling the null values
df_filled = df_dedup.na.fill({
    "Outlet_Size": "Unknown",
    "Item_Weight": avg_weight
})

In [0]:
# varifying null values are removed...
df_filled.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_filled.columns
]).display()

Item_Identifier,item_weight,item_fat_content,item_visibility,item_type,item_mrp,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,0,0,0,0,0,0,0,0,0,0,0


### Transformation 9: Create Visibility Categories

Used `withColumn()` and conditional logic with `when()` to create a new column called `visibility_flag`. Items were categorized as Low, Medium, or High based on their visibility values, making it easier to analyze product exposure and compare sales performance across visibility levels.

**Operations Used:** `withColumn()`, `when()`, `otherwise()`

In [0]:
df_visibility = df_filled.withColumn(
    "visibility_flag",
    when(col("Item_Visibility") < 0.05, "Low")
    .when(col("Item_Visibility") < 0.15, "Medium")
    .otherwise("High")
)

df_visibility.limit(5).display()

Item_Identifier,item_weight,item_fat_content,item_visibility,item_type,item_mrp,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,visibility_flag
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138,Low
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Low
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27,Low
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,Unknown,Tier 3,Grocery Store,732.38,Low
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Low


### Transformation 10: Aggregate Sales by Outlet Type

Used `groupBy()` and `agg()` to summarize sales performance across different outlet types. Calculated the total number of records, total sales, average sales, and average product price (MRP) for each outlet category. This helps identify which outlet types contribute the most to overall business revenue.

**Operations Used:** `groupBy()`, `agg()`, `count()`, `sum()`, `avg()`, `round()`

In [0]:
outlet_summary = (
    df_visibility
    .groupBy("Outlet_Type")
    .agg(
        count("*").alias("total_records"),
        round(sum("Item_Outlet_Sales"), 2).alias("total_sales"),
        round(avg("Item_Outlet_Sales"), 2).alias("avg_sales"),
        round(avg("Item_MRP"), 2).alias("avg_mrp")
    )
)

outlet_summary.display()

Outlet_Type,total_records,total_sales,avg_sales,avg_mrp
Supermarket Type1,5577,1.291734226E7,2316.18,141.21
Supermarket Type2,928,1851822.83,1995.5,141.68
Grocery Store,1083,368034.27,339.83,140.29
Supermarket Type3,935,3453926.05,3694.04,139.8


### Transformation 11: Calculate Outlet Age

Used `withColumn()` to create a new feature called `outlet_age` by subtracting the outlet establishment year from the current year (2026). This feature helps analyze whether older outlets perform differently from newer outlets in terms of sales and customer reach.

**Operations Used:** `withColumn()`, `col()`, `lit()`

In [0]:
df_outlet_age = df_visibility.withColumn(
    "outlet_age",
    lit(2026) - col("Outlet_Establishment_Year")
)
df_outlet_age.limit(5).display()

Item_Identifier,item_weight,item_fat_content,item_visibility,item_type,item_mrp,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,visibility_flag,outlet_age
FDA15,9.3,Low Fat,0.016047301,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.138,Low,27
DRC01,5.92,Regular,0.019278216,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Low,17
FDN15,17.5,Low Fat,0.016760075,Meat,141.618,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.27,Low,27
FDX07,19.2,Regular,0.0,Fruits and Vegetables,182.095,OUT010,1998,Unknown,Tier 3,Grocery Store,732.38,Low,28
NCD19,8.93,Low Fat,0.0,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Low,39


### Transformation 12: Aggregate Sales by Item Type

Used `groupBy()` and `agg()` to analyze sales performance across different product categories. Calculated the total number of items, total sales, average sales, and average MRP for each item type. This helps identify the most profitable product categories and understand their contribution to overall revenue.

**Operations Used:** `groupBy()`, `agg()`, `count()`, `sum()`, `avg()`, `round()`

In [0]:
item_type_summary = (
    df_outlet_age
    .groupBy("Item_Type")
    .agg(
        count("*").alias("total_items"),
        round(sum("Item_Outlet_Sales"), 2).alias("total_sales"),
        round(avg("Item_Outlet_Sales"), 2).alias("avg_sales"),
        round(avg("Item_MRP"), 2).alias("avg_mrp")
    )
)

item_type_summary.limit(7).display()

Item_Type,total_items,total_sales,avg_sales,avg_mrp
Dairy,682,1522594.05,2232.54,148.5
Soft Drinks,445,892897.72,2006.51,131.49
Meat,425,917565.61,2158.98,139.88
Fruits and Vegetables,1232,2820059.82,2289.01,144.58
Household,910,2055493.71,2258.78,149.42
Baking Goods,648,1265525.34,1952.97,126.38
Snack Foods,1200,2732786.09,2277.32,146.19
